# 03 - Cleaning UN DESA Destination Stock Data

This notebook cleans and standardizes UN DESA migration stock data for migrants originating from Cameroon.

Analytical role:

- supports Q1 on destination countries
- keeps migrant stock indicators separate from flow indicators
- prepares a standardized file used later to build `destinations_master.csv`

The cleaning process focuses on keeping the dataset reproducible, simple to inspect and compatible with the project-wide analytical schema.


In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path("..")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

GLOBAL_PATH = DATA_RAW / "global"
CLEANED_PATH = DATA_PROCESSED / "cleaned"

CLEANED_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
undesa = pd.read_csv(GLOBAL_PATH / "undesa_cameroon_global_destinations.csv")
undesa.head()

In [ ]:
undesa.columns.tolist()

In [ ]:
undesa = undesa[undesa["Origin"] == "Cameroon"].copy()
undesa

In [ ]:
for col in ["Total", "Male", "Female"]:
    undesa[col] = undesa[col].astype(str).str.replace(" ", "", regex=False)
    undesa[col] = pd.to_numeric(undesa[col], errors="coerce")
undesa

In [ ]:
sorted(undesa["Year"].unique())

In [ ]:
undesa["Destination"].nunique()

In [ ]:
undesa["Destination"].drop_duplicates().sort_values().tolist()[:50]

In [ ]:
aggregates_to_remove = [
    "World",
    "More developed regions",
    "Less developed regions",
    "Less developed regions, excluding least developed countries",
    "Less developed regions, excluding China",

    "AFRICA",
    "ASIA",
    "EUROPE",
    "OCEANIA",
    "NORTHERN AMERICA",
    "LATIN AMERICA AND THE CARIBBEAN",

    "Africa",
    "Asia",
    "Europe and Northern America",
    "Latin America and the Caribbean",

    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "Central and Southern Asia",
    "Eastern and South-Eastern Asia",

    "Eastern Africa",
    "Middle Africa",
    "Northern Africa",
    "Southern Africa",
    "Western Africa",

    "Caribbean",
    "Central America",
    "South America",

    "Central Asia",
    "Eastern Asia",
    "Southern Asia",
    "South-Eastern Asia",
    "Western Asia",

    "Eastern Europe",
    "Northern Europe",
    "Southern Europe",
    "Western Europe",

    "Australia/New Zealand",
    "Oceania (excluding Australia and New Zealand)",
    "Melanesia",
    "Micronesia",
    "Polynesia*",

    "Least developed countries",
    "Land-locked Developing Countries (LLDC)",
    "Small Island Developing States (SIDS)",

    "Low-income countries",
    "Lower-middle-income countries",
    "Upper-middle-income countries",
    "High-income countries",
    "Middle-income countries",
    "Low-and-middle-income countries",
    "Low-and-Lower-middle-income countries",
    "High-and-upper-middle-income countries",
    "No income group available",
]

undesa_clean = undesa[
    ~undesa["Destination"].isin(aggregates_to_remove)
].copy()

undesa_clean


In [ ]:
undesa_clean = undesa_clean[undesa_clean["Year"].isin([2015, 2020, 2024])].copy()
sorted(undesa_clean["Year"].unique())
undesa_clean

In [ ]:
undesa_clean = undesa_clean.rename(columns={
    "Destination": "destination_country",
    "Location code of destination": "destination_code",
    "Origin": "origin_country",
    "Location code of origin": "origin_code",
    "Year": "year",
    "Total": "total_migrants",
    "Male": "male_migrants",
    "Female": "female_migrants"
})
undesa_clean

In [ ]:
undesa_clean["source"] = "UN DESA"
undesa_clean["dataset"] = "international_migrant_stock"
undesa_clean["measure_type"] = "stock"

undesa_clean["period"] = undesa_clean["year"].map({
    2015: "pre_covid",
    2020: "covid",
    2024: "post_covid"
})
undesa_clean

In [ ]:
undesa_clean.info()
undesa_clean.head()
undesa_clean.isna().sum()

In [ ]:
undesa_clean.to_csv(
    CLEANED_PATH / "undesa_destinations_cleaned.csv",
    index=False
)
undesa_clean = undesa_clean.reset_index(drop=True)
undesa_clean